# Sentence-BERT — Sentence Embeddings using Siamese BERT-Networks (2019)

_Papers · Transformers & LLMs_

**Fine-tune BERT in a siamese setup so a whole sentence becomes one vector whose cosine similarity is meaningful — turning 65-hour pairwise search into a 5-second nearest-vector lookup.**

---

This notebook is a practice scaffold for the **Sentence-BERT — Sentence Embeddings using Siamese BERT-Networks (2019)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# === 0. Worked example: cosine of two sentence embeddings (matches the lesson). ===
u = torch.tensor([2., 1., 0., 1.])
v = torch.tensor([1., 2., 1., 0.])
w = torch.tensor([-1., 0., 2., -1.])      # an "unrelated" vector
def cosine(a, b):                          # math owner: concept dl-cosine-similarity
    return (a @ b) / (a.norm() * b.norm())
print("cos(u,v) =", round(cosine(u, v).item(), 4))   # 0.6667  (similar)
print("cos(u,w) =", round(cosine(u, w).item(), 4))   # -0.5    (dissimilar)
print("cos(u,2v)=", round(cosine(u, 2*v).item(), 4)) # 0.6667  (cosine ignores length)

# === 1. A tiny SIAMESE sentence encoder: shared weights, pool to one vector. ===
# pooling = "mean" (paper default) or "cls" (token 0) -- the ablation toggle (Section 6, Table 6).
VOCAB, D, MAXLEN = 24, 32, 6
class SentenceEncoder(nn.Module):
    def __init__(self, pooling="mean"):
        super().__init__()
        self.pooling = pooling
        self.embed = nn.Embedding(VOCAB, D)
        self.pos   = nn.Parameter(torch.randn(MAXLEN, D) * 0.02)         # learned position
        self.block = nn.TransformerEncoderLayer(d_model=D, nhead=4, dim_feedforward=64,
                                                batch_first=True)        # a BERT-style block (imported)
    def forward(self, tokens):              # tokens: (B, L)
        x = self.embed(tokens) + self.pos[:tokens.shape[1]]
        h = self.block(x)                   # (B, L, D) per-token vectors
        if self.pooling == "mean":
            return h.mean(dim=1)            # MEAN-pool: average all tokens  (paper default)
        else:
            return h[:, 0]                  # CLS-pool: take token 0          (ablation)

# The SAME encoder is called on each sentence -> that is what makes it "siamese".
def embed_pair(enc, sa, sb):
    return enc(sa), enc(sb)

# === 2. Toy data: paraphrases share words; unrelated pairs do not. ===
# Vocab ids 1..23 are "words"; a sentence is a length-6 id sequence. A paraphrase = same word
# multiset, shuffled; an unrelated sentence = a disjoint set of words.
def make_dataset(n=256):
    A, Bp, Bn = [], [], []
    for _ in range(n):
        words = torch.randint(1, 12, (MAXLEN,))           # topic-A words 1..11
        A.append(words)
        Bp.append(words[torch.randperm(MAXLEN)])          # paraphrase: shuffle same words
        Bn.append(torch.randint(12, VOCAB, (MAXLEN,)))    # unrelated: disjoint words 12..23
    A, Bp, Bn = map(lambda L: torch.stack(L), (A, Bp, Bn))
    # pairs: (A, Bp, label=1) and (A, Bn, label=0)
    left  = torch.cat([A, A], 0)
    right = torch.cat([Bp, Bn], 0)
    label = torch.cat([torch.ones(n), torch.zeros(n)], 0)
    return left, right, label

LEFT, RIGHT, LABEL = make_dataset()

# === 3. Train with the REGRESSION objective: MSE( cos(u,v), gold label ).  (Section 3) ===
def train(pooling, steps=400, lr=3e-3):
    torch.manual_seed(0)
    enc = SentenceEncoder(pooling=pooling)
    opt = torch.optim.Adam(enc.parameters(), lr=lr)
    for s in range(steps):
        u, v = embed_pair(enc, LEFT, RIGHT)
        pred = F.cosine_similarity(u, v, dim=1)           # cos in [-1, 1]
        loss = F.mse_loss(pred, LABEL)                    # fit cosine to 1 (para) / 0 (unrelated)
        opt.zero_grad(); loss.backward(); opt.step()
        if s % 100 == 0 or s == steps - 1:
            with torch.no_grad():
                para = pred[LABEL == 1].mean().item()
                unre = pred[LABEL == 0].mean().item()
            print(f"  step {s:3d}  loss {loss.item():.4f}  cos(para) {para:.3f}  cos(unrel) {unre:.3f}  gap {para-unre:.3f}")
    return enc

print("\n--- MEAN pooling (paper default) ---")
enc_mean = train("mean")
print("--- CLS pooling (ablation, Section 6 / Table 6) ---")
enc_cls = train("cls")

# === 4. Semantic-search demo: encode a corpus once, rank by cosine to a query. ===
# Build 4 short "sentences" over the same vocab; query is a paraphrase of corpus sentence 0.
corpus = torch.stack([
    torch.tensor([3, 5, 7, 9, 2, 4]),     # sent 0
    torch.tensor([15, 18, 21, 13, 16, 19]),  # sent 1 (different topic)
    torch.tensor([6, 8, 10, 11, 1, 3]),   # sent 2
    torch.tensor([20, 14, 22, 17, 23, 12]),  # sent 3 (different topic)
])
query = torch.tensor([[9, 7, 2, 5, 4, 3]])    # a shuffle of sent 0's words -> should rank sent 0 first
with torch.no_grad():
    cvecs = enc_mean(corpus)                  # encode corpus ONCE
    qvec  = enc_mean(query)
    sims  = F.cosine_similarity(qvec, cvecs, dim=1)   # then just cosine -- the fast lookup
order = sims.argsort(descending=True)
print("\nsemantic search (query is a paraphrase of corpus sent 0):")
for rank, i in enumerate(order.tolist()):
    print(f"  #{rank+1}: corpus sent {i}   cos {sims[i].item():.3f}")
# Top hit should be sent 0 (highest cosine) -- the encode-once + cosine pattern from Section 7.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_After siamese fine-tuning on toy paraphrase/unrelated pairs, how well does cosine separate the two groups — and does MEAN pooling beat CLS pooling, as the paper's Table 6 ablation reports?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

VOCAB, D, MAXLEN = 24, 32, 6
class Enc(nn.Module):
    def __init__(self, pooling):
        super().__init__(); self.pooling = pooling
        self.embed = nn.Embedding(VOCAB, D)
        self.pos   = nn.Parameter(torch.randn(MAXLEN, D) * 0.02)
        self.block = nn.TransformerEncoderLayer(D, 4, 64, batch_first=True)
    def forward(self, t):
        h = self.block(self.embed(t) + self.pos[:t.shape[1]])
        return h.mean(1) if self.pooling == "mean" else h[:, 0]

def make(n=256):
    A, Bp, Bn = [], [], []
    for _ in range(n):
        wds = torch.randint(1, 12, (MAXLEN,))
        A.append(wds); Bp.append(wds[torch.randperm(MAXLEN)])
        Bn.append(torch.randint(12, VOCAB, (MAXLEN,)))
    A, Bp, Bn = map(torch.stack, (A, Bp, Bn))
    left  = torch.cat([A, A], 0); right = torch.cat([Bp, Bn], 0)
    label = torch.cat([torch.ones(n), torch.zeros(n)], 0)
    return left, right, label

LEFT, RIGHT, LABEL = make()
def run(pooling, steps=400):
    torch.manual_seed(0)
    enc = Enc(pooling); opt = torch.optim.Adam(enc.parameters(), lr=3e-3); gaps = []
    for s in range(steps):
        pred = F.cosine_similarity(enc(LEFT), enc(RIGHT), dim=1)
        loss = F.mse_loss(pred, LABEL)
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            gaps.append((pred[LABEL == 1].mean() - pred[LABEL == 0].mean()).item())
    return gaps

mean_gaps = run("mean")
cls_gaps  = run("cls")
idx = list(range(0, 400, 40)) + [399]
print("MEAN pool gap:", [[i, round(mean_gaps[i], 3)] for i in idx])
print("CLS  pool gap:", [[i, round(cls_gaps[i], 3)] for i in idx])
# MEAN opens a wider paraphrase-vs-unrelated cosine gap than CLS -- matches Table 6's MEAN > CLS ordering.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation (Table 6). Your siamese encoder is trained with MEAN pooling and separates
            paraphrase pairs (high cosine) from unrelated pairs (low cosine). Now swap to CLS pooling
            (use only token 0's vector) and retrain with everything else identical. What do you expect to
            happen to the paraphrase-vs-unrelated cosine gap, and what does the paper report?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change exactly one line: the pooling, from mean over token vectors to take token 0. Keep encoder depth/width, optimizer, data, and seed identical. — _An honest ablation changes one thing &mdash; the pooling &mdash; so any difference is attributable to it._
- Retrain and measure the gap = (mean cosine on paraphrase pairs) &minus; (mean cosine on unrelated pairs) for each pooling. — _A larger gap means the embedding space separates meaning better &mdash; the quantity the paper's STS score reflects._
- Compare to Table 6: MEAN $=80.78$ beats CLS $=79.80$ and MAX $=79.07$. — _MEAN aggregates evidence from every token; one slot (CLS) is a thinner summary, especially without BERT's full pretraining behind it._

**Answer:** MEAN pooling gives the wider paraphrase-vs-unrelated gap; CLS is a bit worse. The paper's
                 Table 6 ranks them MEAN ($80.78$) > CLS ($79.80$) > MAX ($79.07$), which is why MEAN is
                 the default. Intuition: averaging every token's vector pools evidence from the whole sentence,
                 while one designated slot is a thinner summary. Our CODEVIZ shows the same qualitative ordering
                 on toy data (our numbers, not the paper's).

</details>

**Problem 2.** In the worked example, $u=[2,1,0,1]$ and $v=[1,2,1,0]$ gave $\cos(u,v)\approx0.667$, while the
            unrelated $w=[-1,0,2,-1]$ gave $\cos(u,w)=-0.5$. If you doubled $v$ to $[2,4,2,0]$, what
            happens to $\cos(u,v)$, and why does that property make cosine a good choice for sentence vectors?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recompute with $v'=2v$: dot product $u\cdot v' = 2\cdot(u\cdot v)=8$; length $\lVert v'\rVert = 2\lVert v\rVert = 2\sqrt6$. — _Scaling a vector scales both the dot product and its norm by the same factor._
- So $\cos(u,v') = \dfrac{8}{\sqrt6\,\cdot 2\sqrt6} = \dfrac{8}{12} = 0.667$ &mdash; unchanged. — _The factor $2$ cancels between numerator and denominator: cosine is invariant to vector length._
- Conclude cosine measures only direction (angle), not magnitude. — _Sentence-vector magnitude can vary for reasons unrelated to meaning, so ignoring it is desirable._

**Answer:** $\cos(u,v)$ is unchanged at $0.667$: scaling $v$ by $2$ multiplies both the dot
                 product and $\lVert v\rVert$ by $2$, and they cancel. Cosine looks only at the angle
                 between vectors, not their length. That is exactly why SBERT compares sentences with cosine:
                 two sentences with the same meaning should score high regardless of incidental magnitude
                 differences. (Full treatment in concept dl-cosine-similarity.)

</details>

**Problem 3.** The paper says finding the most similar pair among 10,000 sentences takes ~65 hours with BERT but
            ~5 seconds with SBERT (abstract). Where does the speedup come from, given both ultimately compute
            similarities?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count BERT's work: it scores a pair only by running BOTH sentences through the network together, so each of the ~50 million pairs is a fresh forward pass. — _BERT is a cross-encoder &mdash; no reusable per-sentence vector exists, so work is $O(n^2)$ forward passes._
- Count SBERT's work: encode each of the 10,000 sentences ONCE into a vector ($n$ forward passes), then compare any two by cosine. — _Cosine on stored vectors is cheap arithmetic ($\sim$0.01s for the whole comparison step), so the cost is $n$ encodes, not $n^2$._
- Conclude the win is moving the expensive network out of the inner loop: encode-once, then fast vector math. — _This is the enabling trick behind semantic search, clustering, and vector databases._

**Answer:** BERT is a cross-encoder: it has no reusable sentence vector, so it must re-run the full
                 network on every pair &mdash; about 50 million joint forward passes ($O(n^2)$). SBERT encodes
                 each sentence once into a fixed vector ($n$ passes) and then compares any two with a
                 cheap cosine. The expensive network leaves the inner loop, turning $O(n^2)$ network calls into
                 $n$ encodes plus fast vector arithmetic &mdash; 65 hours becomes ~5 seconds. This encode-once
                 pattern is the foundation of modern semantic search and RAG.

</details>